tUNING eRROR aNALYSIS

In [1]:
import pandas as pd, numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, GridSearchCV, cross_val_predict

df = pd.read_csv("../data/resumes_processed.csv")
X, y = df["processed"], df["Category"]
cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

In [2]:
pipe = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True)),
    ("clf", LinearSVC()),
])
param_grid = {
    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "clf__C": [0.1, 1, 10],
}
grid = GridSearchCV(pipe, param_grid, cv=cv, scoring="accuracy", n_jobs=-1)
grid.fit(X, y)
print(grid.best_params_, round(grid.best_score_, 3))

{'clf__C': 10, 'tfidf__min_df': 2, 'tfidf__ngram_range': (1, 2)} 0.898


In [3]:
pred = cross_val_predict(grid.best_estimator_, X, y, cv=cv)
cm = pd.crosstab(y, pred)
errors = [(t, p, cm.loc[t, p]) for t in cm.index for p in cm.columns
          if t != p and cm.loc[t, p] > 0]
for t, p, n in sorted(errors, key=lambda e: -e[2]):
    print(f"{t} -> {p}: {n}")

Web Designing -> Java Developer: 3
Automation Testing -> Electrical Engineering: 1
Blockchain -> Data Science: 1
Business Analyst -> Automation Testing: 1
Database -> Data Science: 1
DevOps Engineer -> Database: 1
DevOps Engineer -> Java Developer: 1
Mechanical Engineer -> Business Analyst: 1
Network Security Engineer -> Database: 1
PMO -> DevOps Engineer: 1
SAP Developer -> ETL Developer: 1
Testing -> Automation Testing: 1
Testing -> Electrical Engineering: 1
Testing -> Java Developer: 1
Web Designing -> DotNet Developer: 1


In [4]:
import joblib
joblib.dump(grid.best_estimator_, "../app/model.joblib")

['../app/model.joblib']

In [5]:
for t in df[df["Category"] == "Web Designing"]["processed"]:
    print(t[:300], "\n---")

technical web technology angular j html5 css3 sas bootstrap jquery javascript software bracket visual studio photoshop visual studio code 2015 b.e cse g.h.raisoni college engineering 2009 photography competition click state board college magazine ocean web designer web designer trust system software 
---
b.c.a bachelor computer application university h.s.c university s.s.c university web designing developer php developer exposys pvt ltd exposys pvt ltd technical web development html5 css3 bootstrap php ajax jquery javascript database mysql development tool notepad++ sublime text2 framework codeignit 
---
2016 b.sc information technology university 2012 hsc allahabad uttar pradesh allahabad university 2010 ssc dot net allahabad uttar pradesh allahabad university web designer developer trainer web designer developer web design 12 php 12 nettech salary 12k personal interest listening music surfing net  
---
language basic java basic web technology html5 css3 bootstrap javascript jquery co